## Experiment No: 7
## Experiment Title: Named Entity Recognition (NER) and Evaluation

**Name:** Himanshu Jadhav  
**Roll Number:** TE-32

### Step 1: Import Libraries

In [1]:
import re
import spacy
import pandas as pd
from tabulate import tabulate
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

print("Libraries imported successfully.")

Libraries imported successfully.


### Step 2: Sample Text with Named Entities

In [2]:
text = ("Elon Musk founded SpaceX in California in the year 2002. "
        "He also leads Tesla, and previously worked with Google in India. "
        "The next major announcement is expected on 15 August 2026.")

print(text)

Elon Musk founded SpaceX in California in the year 2002. He also leads Tesla, and previously worked with Google in India. The next major announcement is expected on 15 August 2026.


### Step 3: Gold Standard Annotation (Manually Prepared)

In [3]:
# This is our manually created ground truth for evaluation
gold_standard = [
    ("Elon Musk", "PERSON"),
    ("SpaceX", "ORG"),
    ("California", "GPE"),
    ("2002", "DATE"),
    ("Tesla", "ORG"),
    ("Google", "ORG"),
    ("India", "GPE"),
    ("15 August 2026", "DATE")
]

gold_df = pd.DataFrame(gold_standard, columns=["Entity", "Type"])
print(tabulate(gold_df, headers="keys", tablefmt="psql"))

+----+----------------+--------+
|    | Entity         | Type   |
|----+----------------+--------|
|  0 | Elon Musk      | PERSON |
|  1 | SpaceX         | ORG    |
|  2 | California     | GPE    |
|  3 | 2002           | DATE   |
|  4 | Tesla          | ORG    |
|  5 | Google         | ORG    |
|  6 | India          | GPE    |
|  7 | 15 August 2026 | DATE   |
+----+----------------+--------+


### Step 4: Rule-Based NER using Gazetteer + Regex

In [4]:
# A small gazetteer (predefined list) of known organizations and locations
known_orgs   = ["SpaceX", "Tesla", "Google"]
known_places = ["California", "India"]

rule_based_entities = []

for org in known_orgs:
    if org in text:
        rule_based_entities.append((org, "ORG"))

for place in known_places:
    if place in text:
        rule_based_entities.append((place, "GPE"))

date_pattern = r'\b\d{1,2}\s\w+\s\d{4}\b|\b\d{4}\b'
dates_found = re.findall(date_pattern, text)
for d in dates_found:
    rule_based_entities.append((d, "DATE"))

rule_df = pd.DataFrame(rule_based_entities, columns=["Entity", "Type"])
print("Rule-Based NER Output:")
print(tabulate(rule_df, headers='keys', tablefmt='psql'))

Rule-Based NER Output:
+----+----------------+--------+
|    | Entity         | Type   |
|----+----------------+--------|
|  0 | SpaceX         | ORG    |
|  1 | Tesla          | ORG    |
|  2 | Google         | ORG    |
|  3 | California     | GPE    |
|  4 | India          | GPE    |
|  5 | 2002           | DATE   |
|  6 | 15 August 2026 | DATE   |
+----+----------------+--------+


### Step 5: spaCy Pretrained NER

In [5]:
doc = nlp(text)

spacy_entities = [(ent.text, ent.label_) for ent in doc.ents]
spacy_df       = pd.DataFrame(spacy_entities, columns=["Entity", "Type"])

print("spaCy NER Output:")
print(tabulate(spacy_df, headers='keys', tablefmt='psql'))

spaCy NER Output:
+----+----------------+--------+
|    | Entity         | Type   |
|----+----------------+--------|
|  0 | Elon Musk      | PERSON |
|  1 | California     | GPE    |
|  2 | the year 2002  | DATE   |
|  3 | Tesla          | ORG    |
|  4 | Google         | ORG    |
|  5 | India          | GPE    |
|  6 | 15 August 2026 | DATE   |
+----+----------------+--------+


### Step 6: Visualize Entities with displaCy

In [6]:
displacy.render(doc, style="ent", jupyter=True)

### Step 7: Evaluate Rule-Based NER against Gold Standard

In [7]:
def evaluate(predicted, gold):
    predicted_set = set(predicted)
    gold_set      = set(gold)

    tp = len(predicted_set & gold_set)
    fp = len(predicted_set - gold_set)
    fn = len(gold_set - predicted_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score  = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return tp, fp, fn, precision, recall, f1_score

rb_tp, rb_fp, rb_fn, rb_p, rb_r, rb_f1 = evaluate(rule_based_entities, gold_standard)


print("Rule-Based NER Evaluation:")
print("TP:", rb_tp, "FP:", rb_fp, "FN:", rb_fn)
print(f"Precision: {rb_p:.2f}, Recall: {rb_r:.2f}, F1-score: {rb_f1:.2f}")

Rule-Based NER Evaluation:
TP: 7 FP: 0 FN: 1
Precision: 1.00, Recall: 0.88, F1-score: 0.93


### Step 8: Evaluate spaCy NER against Gold Standard

In [8]:
sp_tp, sp_fp, sp_fn, sp_p, sp_r, sp_f1 = evaluate(spacy_entities, gold_standard)

print("spaCy NER Evaluation:")
print("TP:", sp_tp, "FP:", sp_fp, "FN:", sp_fn)
print(f"Precision: {sp_p:.2f}, Recall: {sp_r:.2f}, F1-score: {sp_f1:.2f}")

spaCy NER Evaluation:
TP: 6 FP: 1 FN: 2
Precision: 0.86, Recall: 0.75, F1-score: 0.80


### Step 9: Comparison Table

In [9]:
comparison_df = pd.DataFrame({
    "Approach"  : ["Rule-Based", "spaCy (Statistical)"],
    "TP"        : [rb_tp, sp_tp],
    "FP"        : [rb_fp, sp_fp],
    "FN"        : [rb_fn, sp_fn],
    "Precision" : [round(rb_p, 2), round(sp_p, 2)],
    "Recall"    : [round(rb_r, 2), round(sp_r, 2)],
    "F1-score"  : [round(rb_f1, 2), round(sp_f1, 2)]
})

print(tabulate(comparison_df, headers='keys', tablefmt='psql'))

+----+---------------------+------+------+------+-------------+----------+------------+
|    | Approach            |   TP |   FP |   FN |   Precision |   Recall |   F1-score |
|----+---------------------+------+------+------+-------------+----------+------------|
|  0 | Rule-Based          |    7 |    0 |    1 |        1    |     0.88 |       0.93 |
|  1 | spaCy (Statistical) |    6 |    1 |    2 |        0.86 |     0.75 |       0.8  |
+----+---------------------+------+------+------+-------------+----------+------------+


### Final Output

In [10]:
print("Experiment Completed Successfully\n")
print(tabulate(comparison_df, headers='keys', tablefmt='fancy_grid'))

Experiment Completed Successfully

╒════╤═════════════════════╤══════╤══════╤══════╤═════════════╤══════════╤════════════╕
│    │ Approach            │   TP │   FP │   FN │   Precision │   Recall │   F1-score │
╞════╪═════════════════════╪══════╪══════╪══════╪═════════════╪══════════╪════════════╡
│  0 │ Rule-Based          │    7 │    0 │    1 │        1    │     0.88 │       0.93 │
├────┼─────────────────────┼──────┼──────┼──────┼─────────────┼──────────┼────────────┤
│  1 │ spaCy (Statistical) │    6 │    1 │    2 │        0.86 │     0.75 │       0.8  │
╘════╧═════════════════════╧══════╧══════╧══════╧═════════════╧══════════╧════════════╛
